# Milvus数据库的DML操作，需要嵌入模型，
## 我们这里使用bge-m3:latest模型，它支持1024维度

## 初始化嵌入模型

In [2]:
from langchain.embeddings import init_embeddings
from pymilvus.client.types import MetricType

em_model = init_embeddings(
    ## 我们这里使用bge-m3:latest模型，它支持1024维度
    model="ollama:bge-m3:latest"
)


## 准备Collection
### 创建一个名为docs的collection

In [11]:
from pymilvus import MilvusClient
from rich import print as rprint
client = MilvusClient(host='localhost', port=19530)
# 切换数据库
client.use_database("rag_tutorial")
# 创建collection
col_name = "docs"
client.create_collection(
    collection_name=col_name,
    dimension=1024, # 给bge-m3:latest模型使用必须是1024维
    metric_type="COSINE",
)

# 查看一下
rprint(client.list_collections())

['docs']

### 查看Collection元数据

In [13]:
meta_data = client.describe_collection(collection_name=col_name, )
rprint(meta_data)

{
    'collection_name': 'docs',
    'auto_id': False,
    'num_shards': 1,
    'description': '',
    'fields': [
        {
            'field_id': 100,
            'name': 'id',
            'description': '',
            'type': <DataType.INT64: 5>,
            'params': {},
            'is_primary': True
        },
        {
            'field_id': 101,
            'name': 'vector',
            'description': '',
            'type': <DataType.FLOAT_VECTOR: 101>,
            'params': {'dim': 1024}
        }
    ],
    'functions': [],
    'aliases': [],
    'collection_id': 470399118321993632,
    'consistency_level': 2,
    'properties': {'timezone': 'UTC', 'namespace.sharding.enabled': 'false', 'max_field_id': '102'},
    'num_partitions': 1,
    'enable_dynamic_field': True,
    'enable_namespace': False,
    'created_timestamp': 470403360512540690,
    'update_timestamp': 470403360512540690
}

## 准备数据
### 准备原始数据

In [15]:
# 准备测试数据
texts = [
    "LangChain 是一个用于构建 LLM 应用的开发框架。",
    "Milvus 是一个适合 AI 应用的向量数据库。",
    "RAG 的核心是先检索相关知识，再让大模型生成答案。",
    "Docker Desktop 可以方便地在本地运行 Milvus Standalone。"
]

### 生成嵌入向量

In [17]:
vectors = em_model.embed_documents(texts)

### 查看生成的嵌入向量

In [18]:
print(len(vectors))
print(len(vectors[0]))
print(vectors[0][:5])


4
1024
[-0.0072477185, -0.01241455, 0.006812786, -0.023820743, -0.010981089]


### 封装为可以插入的数据格式

In [19]:
data = [
    {
        "id":i,
        "vector":vectors[i],
        "text":texts[i],
        "source":"demo"
    } for i in range(len(texts))
]

## 写入数据
### 1.插入数据

In [20]:
inserted = client.upsert(
    collection_name=col_name,
    data=data,
)
print("inserted result",inserted)

inserted result {'upsert_count': 4, 'ids': [0, 1, 2, 3]}


### 2.手动flush

Milvus不会第一时间将数据落盘，要看到写入效果，我们手动flush，将数据刷写到磁盘

In [21]:
client.flush(collection_name=col_name)

### 3.查看collection的统计信息

In [22]:
stats = client.get_collection_stats(collection_name=col_name)
print("stats : ",stats)

stats :  {'row_count': 4}


## 查询数据
### 1.扫描数据

In [24]:
# 扫描返回一个迭代器
iterator= client.query_iterator(
    collection_name=col_name,
    filter="",
    output_fields=["*"]
)
# 遍历这个迭代器
i = 0
while True:
    rows = iterator.next()
    if not rows:
        break # 没有数据就跳出循环
    for row in rows:
        print(f"第{i+1}条数据")
        print(f"id:{row['id']},vector:{row['vector'][:6]},text:{row['text']},source:{row['source']}")
        i+=1
# 关闭迭代器
iterator.close()



第1条数据
id:0,vector:[-0.007247718516737223, -0.01241455040872097, 0.006812786217778921, -0.023820742964744568, -0.010981089435517788, 0.003364304779097438],text:LangChain 是一个用于构建 LLM 应用的开发框架。,source:demo
第2条数据
id:1,vector:[-0.02614247426390648, -0.007813695818185806, -0.044614244252443314, 0.027146827429533005, -0.03346766158938408, -0.023296315222978592],text:Milvus 是一个适合 AI 应用的向量数据库。,source:demo
第3条数据
id:2,vector:[-0.016585901379585266, -0.008772866800427437, 0.007794947363436222, -0.0005849088774994016, -0.007447077892720699, -0.021641787141561508],text:RAG 的核心是先检索相关知识，再让大模型生成答案。,source:demo
第4条数据
id:3,vector:[0.01743341237306595, 0.028369862586259842, -0.04740368574857712, -0.008444947190582752, -0.020675532519817352, 0.025601768866181374],text:Docker Desktop 可以方便地在本地运行 Milvus Standalone。,source:demo


### 通过主键查询数据
### 使用get方法，可以传递一个列表ids

In [26]:
rows = client.get(
    collection_name=col_name,
    ids=[0,1,2]
)

for idx, row in enumerate(rows):
    print(f"第{idx+1}条数据：")
    print(f"id:{row['id']},vector:{row['vector'][:6]},text:{row['text']},source:{row['source']}")

第1条数据：
id:0,vector:[-0.007247718516737223, -0.01241455040872097, 0.006812786217778921, -0.023820742964744568, -0.010981089435517788, 0.003364304779097438],text:LangChain 是一个用于构建 LLM 应用的开发框架。,source:demo
第2条数据：
id:1,vector:[-0.02614247426390648, -0.007813695818185806, -0.044614244252443314, 0.027146827429533005, -0.03346766158938408, -0.023296315222978592],text:Milvus 是一个适合 AI 应用的向量数据库。,source:demo
第3条数据：
id:2,vector:[-0.016585901379585266, -0.008772866800427437, 0.007794947363436222, -0.0005849088774994016, -0.007447077892720699, -0.021641787141561508],text:RAG 的核心是先检索相关知识，再让大模型生成答案。,source:demo


### 相似度检索
#### 1.准备查询嵌入

In [27]:
# 相似度检索
query = "什么是向量数据库？"
query_vector = em_model.embed_query(query)
rprint(query_vector[:10])

[
    -0.05891716,
    -0.043307595,
    -0.031699978,
    0.028268743,
    0.002885767,
    0.006640191,
    0.010140234,
    -0.040027887,
    0.015564025,
    -0.0097677745
]

#### 2.检索
##### 调用search方法


In [28]:
results = client.search(
    collection_name=col_name,
    data=[query_vector],
    limit=3,
    output_fields=["text","source","id"]
)

for res in results[0]: # 这个列表是一个e二维列表.她其实只有一个列表也是，所以我们只需要取第一个
    print(res)

{'id': 1, 'distance': 0.7204978466033936, 'entity': {'id': 1, 'text': 'Milvus 是一个适合 AI 应用的向量数据库。', 'source': 'demo'}}
{'id': 0, 'distance': 0.46550601720809937, 'entity': {'id': 0, 'text': 'LangChain 是一个用于构建 LLM 应用的开发框架。', 'source': 'demo'}}
{'id': 2, 'distance': 0.4186750054359436, 'entity': {'id': 2, 'text': 'RAG 的核心是先检索相关知识，再让大模型生成答案。', 'source': 'demo'}}
